In [ ]:
# =========================================================
# SENTENCE-LEVEL EXPERIMENT FRAMEWORK
# Models: SVM, LSTM, LVBERT, HSSC
# =========================================================

import os
import copy
import random
import warnings
import shutil
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# =========================================================
# 1. CONFIG
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = "/content/train_sen.csv"
VAL_PATH = "/content/validation_sen.csv"
TEST_PATH = "/content/test_sen.csv"

TEXT_COL = "sentence_text"
TARGET_COL = "role_label"
GROUP_COL = "article_id"
CLAIM_LABEL = "CLAIM"

LVBERT_MODEL_NAME = "AiLab-IMCS-UL/lvbert"

ARTIFACT_ROOT = "/content/universal_sentence_experiments"
os.makedirs(ARTIFACT_ROOT, exist_ok=True)

FEATURE_SETS = {
    "legacy": {
        "numeric_cols": [
            "sentence_id",
            "sentence_index_in_article",
            "num_sentences_in_article",
            "relative_position",
            "article_word_count",
            "sentence_char_len",
            "claim_group_id",
            "linked_claim_sentence_id",
            "is_primary_claim"
        ],
        "categorical_cols": [
            "position_bucket",
            "source_part",
            "sample_group",
            "group"
        ]
    },
    "safe": {
        "numeric_cols": [
            "sentence_index_in_article",
            "num_sentences_in_article",
            "relative_position",
            "article_word_count",
            "sentence_char_len"
        ],
        "categorical_cols": [
            "position_bucket",
            "source_part"
        ]
    }
}

# ---------- LSTM ----------
LSTM_MAX_TOKENS = 20000
LSTM_MAX_LEN = 80
LSTM_EMBED_DIM = 128
LSTM_UNITS = 64
LSTM_BATCH_SIZE = 32
LSTM_EPOCHS = 15
LSTM_PATIENCE = 3

# ---------- LVBERT ----------
LVBERT_MAX_LEN = 128
LVBERT_BATCH_SIZE = 16
LVBERT_EPOCHS = 5
LVBERT_LR_BERT = 2e-5
LVBERT_LR_HEAD = 1e-3
LVBERT_WEIGHT_DECAY = 0.01
LVBERT_PATIENCE = 2

# =========================================================
# 2. COMMON HELPERS
# =========================================================
def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except Exception:
        pass

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def clean_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def get_experiment_dir(model_type, feature_mode):
    path = os.path.join(ARTIFACT_ROOT, f"{model_type}_{feature_mode}")
    ensure_dir(path)
    return path

def print_header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

def print_eval(y_true, y_pred, label_encoder, split_name):
    all_label_ids = np.arange(len(label_encoder.classes_))
    print(f"\n===== {split_name} =====")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, average="macro", labels=all_label_ids), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, average="weighted", labels=all_label_ids), 4))
    print(classification_report(
        y_true,
        y_pred,
        labels=all_label_ids,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    ))

def save_multiclass_prediction_file(df_input, pred_probs, pred_labels, output_path, label_encoder, cv_mode=False):
    out = df_input.copy()

    pred_label_col = "pred_role_label_cv" if cv_mode else "pred_role_label"
    pred_conf_col = "pred_confidence_cv" if cv_mode else "pred_confidence"

    out[pred_label_col] = label_encoder.inverse_transform(pred_labels)
    out[pred_conf_col] = np.max(pred_probs, axis=1)

    for i, class_name in enumerate(label_encoder.classes_):
        safe_name = str(class_name).lower().replace(" ", "_")
        prob_col = f"pred_prob_{safe_name}_cv" if cv_mode else f"pred_prob_{safe_name}"
        out[prob_col] = pred_probs[:, i]

    out.to_csv(output_path, index=False, encoding="utf-8-sig")
    return out

def save_summary(results_dict, output_path):
    df = pd.DataFrame([results_dict])
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    return df

# =========================================================
# 3. DATA LOADING
# =========================================================
def load_data(feature_mode="legacy"):
    if feature_mode not in FEATURE_SETS:
        raise ValueError(f"Unknown feature_mode: {feature_mode}")

    numeric_cols = FEATURE_SETS[feature_mode]["numeric_cols"].copy()
    categorical_cols = FEATURE_SETS[feature_mode]["categorical_cols"].copy()

    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    test_df = pd.read_csv(TEST_PATH)

    for df in [train_df, val_df, test_df]:
        df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
        df[TARGET_COL] = df[TARGET_COL].astype(str)
        df[GROUP_COL] = pd.to_numeric(df[GROUP_COL], errors="coerce")

    numeric_cols = [c for c in numeric_cols if c in train_df.columns]
    categorical_cols = [c for c in categorical_cols if c in train_df.columns]

    for df in [train_df, val_df, test_df]:
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        for col in categorical_cols:
            df[col] = df[col].astype(str)

    train_df = train_df[
        train_df[TEXT_COL].notna() &
        train_df[TARGET_COL].notna() &
        train_df[GROUP_COL].notna()
    ].copy().reset_index(drop=True)

    val_df = val_df[
        val_df[TEXT_COL].notna() &
        val_df[TARGET_COL].notna() &
        val_df[GROUP_COL].notna()
    ].copy().reset_index(drop=True)

    test_df = test_df[
        test_df[TEXT_COL].notna() &
        test_df[TARGET_COL].notna() &
        test_df[GROUP_COL].notna()
    ].copy().reset_index(drop=True)

    train_df[GROUP_COL] = train_df[GROUP_COL].astype(int)
    val_df[GROUP_COL] = val_df[GROUP_COL].astype(int)
    test_df[GROUP_COL] = test_df[GROUP_COL].astype(int)

    return train_df, val_df, test_df, numeric_cols, categorical_cols

# =========================================================
# 4. LABEL ENCODING
# =========================================================
def encode_labels(train_df, val_df, test_df):
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df[TARGET_COL])
    y_val = label_encoder.transform(val_df[TARGET_COL])
    y_test = label_encoder.transform(test_df[TARGET_COL])
    return label_encoder, y_train, y_val, y_test

# =========================================================
# 5. SVM
# =========================================================
def build_svm_pipeline(numeric_cols, categorical_cols):
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.feature_extraction.text import TfidfVectorizer

    preprocessor = ColumnTransformer(
        transformers=[
            ("text", TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            ), TEXT_COL),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols),
        ],
        remainder="drop"
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(random_state=SEED, max_iter=30000, dual="auto"),
            method="sigmoid",
            cv=5
        ))
    ])
    return model

def run_svm(train_df, val_df, test_df, y_train, y_val, y_test,
            numeric_cols, categorical_cols, label_encoder, feature_mode):

    exp_dir = get_experiment_dir("svm", feature_mode)

    train_oof_path = os.path.join(exp_dir, "train_oof_predictions.csv")
    val_pred_path = os.path.join(exp_dir, "validation_predictions.csv")
    test_pred_path = os.path.join(exp_dir, "test_predictions.csv")
    fold_metrics_path = os.path.join(exp_dir, "fold_metrics.csv")
    summary_path = os.path.join(exp_dir, "summary.csv")

    feature_cols = [TEXT_COL] + numeric_cols + categorical_cols
    X_train = train_df[feature_cols].copy()
    X_val = val_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    groups_train = train_df[GROUP_COL].values
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    n_classes = len(label_encoder.classes_)
    oof_probs = np.zeros((len(train_df), n_classes), dtype=np.float32)
    oof_preds = np.zeros(len(train_df), dtype=np.int32)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups=groups_train), start=1):
        print_header(f"SVM FOLD {fold}/5")

        model = build_svm_pipeline(numeric_cols, categorical_cols)
        model.fit(X_train.iloc[tr_idx], y_train[tr_idx])

        fold_probs = model.predict_proba(X_train.iloc[va_idx])
        fold_preds = np.argmax(fold_probs, axis=1)

        oof_probs[va_idx] = fold_probs
        oof_preds[va_idx] = fold_preds

        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(y_train[va_idx], fold_preds),
            "macro_f1": f1_score(y_train[va_idx], fold_preds, average="macro"),
            "weighted_f1": f1_score(y_train[va_idx], fold_preds, average="weighted")
        })

        print_eval(y_train[va_idx], fold_preds, label_encoder, f"SVM | Fold {fold} validation")

    print_eval(y_train, oof_preds, label_encoder, "SVM | 5-fold CV on TRAIN")

    final_model = build_svm_pipeline(numeric_cols, categorical_cols)
    final_model.fit(X_train, y_train)

    val_probs = final_model.predict_proba(X_val)
    val_preds = np.argmax(val_probs, axis=1)
    print_eval(y_val, val_preds, label_encoder, "SVM | VALIDATION")

    test_probs = final_model.predict_proba(X_test)
    test_preds = np.argmax(test_probs, axis=1)
    print_eval(y_test, test_preds, label_encoder, "SVM | TEST")

    pd.DataFrame(fold_rows).to_csv(fold_metrics_path, index=False, encoding="utf-8-sig")

    save_multiclass_prediction_file(train_df, oof_probs, oof_preds, train_oof_path, label_encoder, cv_mode=True)
    save_multiclass_prediction_file(val_df, val_probs, val_preds, val_pred_path, label_encoder, cv_mode=False)
    save_multiclass_prediction_file(test_df, test_probs, test_preds, test_pred_path, label_encoder, cv_mode=False)

    save_summary({
        "model": f"SVM_{feature_mode}",
        "cv_accuracy": accuracy_score(y_train, oof_preds),
        "cv_macro_f1": f1_score(y_train, oof_preds, average="macro"),
        "cv_weighted_f1": f1_score(y_train, oof_preds, average="weighted"),
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_macro_f1": f1_score(y_val, val_preds, average="macro"),
        "val_weighted_f1": f1_score(y_val, val_preds, average="weighted"),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_macro_f1": f1_score(y_test, test_preds, average="macro"),
        "test_weighted_f1": f1_score(y_test, test_preds, average="weighted")
    }, summary_path)

    print("\nSaved files:")
    print(train_oof_path)
    print(val_pred_path)
    print(test_pred_path)
    print(fold_metrics_path)
    print(summary_path)

    return {
        "train_oof_path": train_oof_path,
        "val_pred_path": val_pred_path,
        "test_pred_path": test_pred_path,
        "summary_path": summary_path
    }

# =========================================================
# 6. LSTM
# =========================================================
def run_lstm(train_df, val_df, test_df, y_train, y_val, y_test,
             numeric_cols, categorical_cols, label_encoder, feature_mode):
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    from tensorflow.keras.callbacks import EarlyStopping

    reset_seeds(SEED)

    exp_dir = get_experiment_dir("lstm", feature_mode)

    train_oof_path = os.path.join(exp_dir, "train_oof_predictions.csv")
    val_pred_path = os.path.join(exp_dir, "validation_predictions.csv")
    test_pred_path = os.path.join(exp_dir, "test_predictions.csv")
    fold_metrics_path = os.path.join(exp_dir, "fold_metrics.csv")
    summary_path = os.path.join(exp_dir, "summary.csv")

    def prepare_numeric(train_part, other_part):
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        x_train_num = imputer.fit_transform(train_part[numeric_cols])
        x_train_num = scaler.fit_transform(x_train_num)

        x_other_num = imputer.transform(other_part[numeric_cols])
        x_other_num = scaler.transform(x_other_num)

        return x_train_num.astype("float32"), x_other_num.astype("float32")

    def build_vectorizer(train_texts):
        vectorizer = layers.TextVectorization(
            max_tokens=LSTM_MAX_TOKENS,
            output_mode="int",
            output_sequence_length=LSTM_MAX_LEN,
            standardize="lower_and_strip_punctuation"
        )
        text_ds = tf.data.Dataset.from_tensor_slices(train_texts).batch(256)
        vectorizer.adapt(text_ds)
        return vectorizer

    def build_model(vectorizer, num_numeric_features, num_classes):
        text_input = layers.Input(shape=(1,), dtype=tf.string, name="text")
        x = vectorizer(text_input)
        x = layers.Embedding(LSTM_MAX_TOKENS, LSTM_EMBED_DIM, mask_zero=True)(x)
        x = layers.Bidirectional(layers.LSTM(LSTM_UNITS))(x)
        x = layers.Dropout(0.3)(x)

        num_input = layers.Input(shape=(num_numeric_features,), dtype=tf.float32, name="num")
        n = layers.Dense(32, activation="relu")(num_input)
        n = layers.Dropout(0.2)(n)

        combined = layers.Concatenate()([x, n])
        combined = layers.Dense(64, activation="relu")(combined)
        combined = layers.Dropout(0.3)(combined)
        output = layers.Dense(num_classes, activation="softmax")(combined)

        model = Model(inputs=[text_input, num_input], outputs=output)
        model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        return model

    def make_inputs(text_series, numeric_array):
        return {
            "text": text_series.astype(str).values,
            "num": numeric_array
        }

    groups_train = train_df[GROUP_COL].values
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    n_classes = len(label_encoder.classes_)
    oof_probs = np.zeros((len(train_df), n_classes), dtype=np.float32)
    oof_preds = np.zeros(len(train_df), dtype=np.int32)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_df, y_train, groups=groups_train), start=1):
        print_header(f"LSTM FOLD {fold}/5")

        fold_train_df = train_df.iloc[tr_idx].copy()
        fold_val_df = train_df.iloc[va_idx].copy()

        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        x_tr_num, x_va_num = prepare_numeric(fold_train_df, fold_val_df)
        vectorizer = build_vectorizer(fold_train_df[TEXT_COL].astype(str).values)
        model = build_model(vectorizer, len(numeric_cols), n_classes)

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=LSTM_PATIENCE,
            restore_best_weights=True
        )

        model.fit(
            make_inputs(fold_train_df[TEXT_COL], x_tr_num),
            y_tr,
            validation_data=(make_inputs(fold_val_df[TEXT_COL], x_va_num), y_va),
            epochs=LSTM_EPOCHS,
            batch_size=LSTM_BATCH_SIZE,
            verbose=0,
            callbacks=[early_stop]
        )

        fold_probs = model.predict(make_inputs(fold_val_df[TEXT_COL], x_va_num), verbose=0)
        fold_preds = np.argmax(fold_probs, axis=1)

        oof_probs[va_idx] = fold_probs
        oof_preds[va_idx] = fold_preds

        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(y_va, fold_preds),
            "macro_f1": f1_score(y_va, fold_preds, average="macro"),
            "weighted_f1": f1_score(y_va, fold_preds, average="weighted")
        })

        print_eval(y_va, fold_preds, label_encoder, f"LSTM | Fold {fold} validation")

    print_eval(y_train, oof_preds, label_encoder, "LSTM | 5-fold CV on TRAIN")

    x_train_num, x_val_num = prepare_numeric(train_df, val_df)
    _, x_test_num = prepare_numeric(train_df, test_df)

    final_vectorizer = build_vectorizer(train_df[TEXT_COL].astype(str).values)
    final_model = build_model(final_vectorizer, len(numeric_cols), n_classes)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=LSTM_PATIENCE,
        restore_best_weights=True
    )

    final_model.fit(
        make_inputs(train_df[TEXT_COL], x_train_num),
        y_train,
        validation_data=(make_inputs(val_df[TEXT_COL], x_val_num), y_val),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        verbose=0,
        callbacks=[early_stop]
    )

    val_probs = final_model.predict(make_inputs(val_df[TEXT_COL], x_val_num), verbose=0)
    val_preds = np.argmax(val_probs, axis=1)
    print_eval(y_val, val_preds, label_encoder, "LSTM | VALIDATION")

    test_probs = final_model.predict(make_inputs(test_df[TEXT_COL], x_test_num), verbose=0)
    test_preds = np.argmax(test_probs, axis=1)
    print_eval(y_test, test_preds, label_encoder, "LSTM | TEST")

    pd.DataFrame(fold_rows).to_csv(fold_metrics_path, index=False, encoding="utf-8-sig")

    save_multiclass_prediction_file(train_df, oof_probs, oof_preds, train_oof_path, label_encoder, cv_mode=True)
    save_multiclass_prediction_file(val_df, val_probs, val_preds, val_pred_path, label_encoder, cv_mode=False)
    save_multiclass_prediction_file(test_df, test_probs, test_preds, test_pred_path, label_encoder, cv_mode=False)

    save_summary({
        "model": f"LSTM_{feature_mode}",
        "cv_accuracy": accuracy_score(y_train, oof_preds),
        "cv_macro_f1": f1_score(y_train, oof_preds, average="macro"),
        "cv_weighted_f1": f1_score(y_train, oof_preds, average="weighted"),
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_macro_f1": f1_score(y_val, val_preds, average="macro"),
        "val_weighted_f1": f1_score(y_val, val_preds, average="weighted"),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_macro_f1": f1_score(y_test, test_preds, average="macro"),
        "test_weighted_f1": f1_score(y_test, test_preds, average="weighted")
    }, summary_path)

    print("\nSaved files:")
    print(train_oof_path)
    print(val_pred_path)
    print(test_pred_path)
    print(fold_metrics_path)
    print(summary_path)

    return {
        "train_oof_path": train_oof_path,
        "val_pred_path": val_pred_path,
        "test_pred_path": test_pred_path,
        "summary_path": summary_path
    }

# =========================================================
# 7. LVBERT
# =========================================================
def run_lvbert(train_df, val_df, test_df, y_train, y_val, y_test,
               numeric_cols, categorical_cols, label_encoder, feature_mode):
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

    reset_seeds(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    exp_dir = get_experiment_dir("lvbert", feature_mode)
    train_oof_path = os.path.join(exp_dir, "train_oof_predictions.csv")
    val_pred_path = os.path.join(exp_dir, "validation_predictions.csv")
    test_pred_path = os.path.join(exp_dir, "test_predictions.csv")
    fold_metrics_path = os.path.join(exp_dir, "fold_metrics.csv")
    summary_path = os.path.join(exp_dir, "summary.csv")

    PipelineNum = SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    PipelineCat = SkPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    def make_struct_preprocessor():
        return ColumnTransformer(
            transformers=[
                ("num", PipelineNum, numeric_cols),
                ("cat", PipelineCat, categorical_cols)
            ],
            remainder="drop"
        )

    class TextStructDataset(Dataset):
        def __init__(self, texts, struct_feats, labels, tokenizer, max_len):
            self.texts = list(texts)
            self.struct_feats = struct_feats.astype(np.float32)
            self.labels = labels.astype(np.int64)
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_len,
                return_tensors="pt"
            )

            item = {
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "struct_feats": torch.tensor(self.struct_feats[idx], dtype=torch.float),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            }

            if "token_type_ids" in enc:
                item["token_type_ids"] = enc["token_type_ids"].squeeze(0)

            return item

    class BertWithStructure(nn.Module):
        def __init__(self, model_name, struct_dim, num_classes, dropout=0.2):
            super().__init__()
            self.bert = AutoModel.from_pretrained(model_name)
            hidden_size = self.bert.config.hidden_size

            self.struct_mlp = nn.Sequential(
                nn.Linear(struct_dim, 64),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            self.classifier = nn.Sequential(
                nn.Linear(hidden_size + 64, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes)
            )

        def forward(self, input_ids, attention_mask, struct_feats, token_type_ids=None):
            if token_type_ids is not None:
                outputs = self.bert(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
            else:
                outputs = self.bert(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

            if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
                pooled = outputs.pooler_output
            else:
                pooled = outputs.last_hidden_state[:, 0, :]

            struct_repr = self.struct_mlp(struct_feats)
            combined = torch.cat([pooled, struct_repr], dim=1)
            logits = self.classifier(combined)
            return logits

    def create_optimizer(model):
        no_decay = ["bias", "LayerNorm.weight"]
        bert_params = []
        head_params = []

        for name, param in model.named_parameters():
            if name.startswith("bert."):
                bert_params.append((name, param))
            else:
                head_params.append((name, param))

        optimizer_grouped_parameters = [
            {
                "params": [p for n, p in bert_params if not any(nd in n for nd in no_decay)],
                "lr": LVBERT_LR_BERT,
                "weight_decay": LVBERT_WEIGHT_DECAY
            },
            {
                "params": [p for n, p in bert_params if any(nd in n for nd in no_decay)],
                "lr": LVBERT_LR_BERT,
                "weight_decay": 0.0
            },
            {
                "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
                "lr": LVBERT_LR_HEAD,
                "weight_decay": LVBERT_WEIGHT_DECAY
            },
            {
                "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
                "lr": LVBERT_LR_HEAD,
                "weight_decay": 0.0
            },
        ]
        return torch.optim.AdamW(optimizer_grouped_parameters)

    def run_epoch(model, loader, optimizer=None, scheduler=None):
        train_mode = optimizer is not None
        model.train() if train_mode else model.eval()

        criterion = nn.CrossEntropyLoss()
        total_loss = 0.0
        all_preds, all_probs, all_labels = [], [], []

        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            struct_feats = batch["struct_feats"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)

            with torch.set_grad_enabled(train_mode):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    struct_feats=struct_feats,
                    token_type_ids=token_type_ids
                )
                loss = criterion(logits, labels)

                if train_mode:
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    if scheduler is not None:
                        scheduler.step()

            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            preds = np.argmax(probs, axis=1)

            total_loss += loss.item() * labels.size(0)
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.detach().cpu().numpy())

        return (
            total_loss / len(loader.dataset),
            np.concatenate(all_preds),
            np.concatenate(all_probs),
            np.concatenate(all_labels)
        )

    def train_one(train_part, val_part, y_tr, y_va):
        tokenizer = AutoTokenizer.from_pretrained(LVBERT_MODEL_NAME)
        struct_preprocessor = make_struct_preprocessor()

        X_train_struct = struct_preprocessor.fit_transform(train_part[numeric_cols + categorical_cols])
        X_val_struct = struct_preprocessor.transform(val_part[numeric_cols + categorical_cols])

        if hasattr(X_train_struct, "toarray"):
            X_train_struct = X_train_struct.toarray()
            X_val_struct = X_val_struct.toarray()

        train_dataset = TextStructDataset(train_part[TEXT_COL].tolist(), X_train_struct, y_tr, tokenizer, LVBERT_MAX_LEN)
        val_dataset = TextStructDataset(val_part[TEXT_COL].tolist(), X_val_struct, y_va, tokenizer, LVBERT_MAX_LEN)

        train_loader = DataLoader(train_dataset, batch_size=LVBERT_BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=LVBERT_BATCH_SIZE, shuffle=False)

        model = BertWithStructure(LVBERT_MODEL_NAME, X_train_struct.shape[1], len(label_encoder.classes_)).to(DEVICE)
        optimizer = create_optimizer(model)

        total_steps = len(train_loader) * LVBERT_EPOCHS
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(1, int(0.1 * total_steps)),
            num_training_steps=total_steps
        )

        best_state = None
        best_val_loss = float("inf")
        patience_counter = 0

        for epoch in range(LVBERT_EPOCHS):
            _ = run_epoch(model, train_loader, optimizer=optimizer, scheduler=scheduler)
            val_loss, val_preds, val_probs, _ = run_epoch(model, val_loader)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= LVBERT_PATIENCE:
                    break

        model.load_state_dict(best_state)
        _, val_preds, val_probs, _ = run_epoch(model, val_loader)
        return model, tokenizer, struct_preprocessor, val_preds, val_probs

    def predict_with_model(model, tokenizer, struct_preprocessor, df, y_enc):
        X_struct = struct_preprocessor.transform(df[numeric_cols + categorical_cols])
        if hasattr(X_struct, "toarray"):
            X_struct = X_struct.toarray()

        dataset = TextStructDataset(df[TEXT_COL].tolist(), X_struct, y_enc, tokenizer, LVBERT_MAX_LEN)
        loader = DataLoader(dataset, batch_size=LVBERT_BATCH_SIZE, shuffle=False)

        _, preds, probs, _ = run_epoch(model, loader)
        return preds, probs

    groups_train = train_df[GROUP_COL].values
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    n_classes = len(label_encoder.classes_)
    oof_probs = np.zeros((len(train_df), n_classes), dtype=np.float32)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_df, y_train, groups=groups_train), start=1):
        print_header(f"LVBERT FOLD {fold}/5")

        train_fold = train_df.iloc[tr_idx].copy()
        val_fold = train_df.iloc[va_idx].copy()

        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        model, tokenizer, struct_preprocessor, va_pred, va_proba = train_one(train_fold, val_fold, y_tr, y_va)
        oof_probs[va_idx] = va_proba

        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(y_va, va_pred),
            "macro_f1": f1_score(y_va, va_pred, average="macro"),
            "weighted_f1": f1_score(y_va, va_pred, average="weighted")
        })

        print_eval(y_va, va_pred, label_encoder, f"LVBERT | Fold {fold} validation")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_preds = np.argmax(oof_probs, axis=1)
    print_eval(y_train, oof_preds, label_encoder, "LVBERT | 5-fold CV on TRAIN")

    final_model, final_tokenizer, final_struct_preprocessor, _, _ = train_one(train_df, val_df, y_train, y_val)

    val_preds, val_probs = predict_with_model(final_model, final_tokenizer, final_struct_preprocessor, val_df, y_val)
    print_eval(y_val, val_preds, label_encoder, "LVBERT | VALIDATION")

    test_preds, test_probs = predict_with_model(final_model, final_tokenizer, final_struct_preprocessor, test_df, y_test)
    print_eval(y_test, test_preds, label_encoder, "LVBERT | TEST")

    pd.DataFrame(fold_rows).to_csv(fold_metrics_path, index=False, encoding="utf-8-sig")

    save_multiclass_prediction_file(train_df, oof_probs, oof_preds, train_oof_path, label_encoder, cv_mode=True)
    save_multiclass_prediction_file(val_df, val_probs, val_preds, val_pred_path, label_encoder, cv_mode=False)
    save_multiclass_prediction_file(test_df, test_probs, test_preds, test_pred_path, label_encoder, cv_mode=False)

    save_summary({
        "model": f"LVBERT_{feature_mode}",
        "cv_accuracy": accuracy_score(y_train, oof_preds),
        "cv_macro_f1": f1_score(y_train, oof_preds, average="macro"),
        "cv_weighted_f1": f1_score(y_train, oof_preds, average="weighted"),
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_macro_f1": f1_score(y_val, val_preds, average="macro"),
        "val_weighted_f1": f1_score(y_val, val_preds, average="weighted"),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_macro_f1": f1_score(y_test, test_preds, average="macro"),
        "test_weighted_f1": f1_score(y_test, test_preds, average="weighted")
    }, summary_path)

    print("\nSaved files:")
    print(train_oof_path)
    print(val_pred_path)
    print(test_pred_path)
    print(fold_metrics_path)
    print(summary_path)

    return {
        "train_oof_path": train_oof_path,
        "val_pred_path": val_pred_path,
        "test_pred_path": test_pred_path,
        "summary_path": summary_path
    }

# =========================================================
# 8. HSSC
# =========================================================
def run_hssc(train_df, val_df, test_df, y_train, y_val, y_test,
             numeric_cols, categorical_cols, label_encoder, feature_mode):

    exp_dir = get_experiment_dir("hssc", feature_mode)

    train_oof_path = os.path.join(exp_dir, "train_oof_predictions.csv")
    val_pred_path = os.path.join(exp_dir, "validation_predictions.csv")
    test_pred_path = os.path.join(exp_dir, "test_predictions.csv")
    fold_metrics_path = os.path.join(exp_dir, "fold_metrics.csv")
    summary_path = os.path.join(exp_dir, "summary.csv")

    lvbert_dir = get_experiment_dir("lvbert", feature_mode)
    lvbert_train_oof_path = os.path.join(lvbert_dir, "train_oof_predictions.csv")
    lvbert_val_path = os.path.join(lvbert_dir, "validation_predictions.csv")
    lvbert_test_path = os.path.join(lvbert_dir, "test_predictions.csv")

    # If lvbert files do not exist, run lvbert automatically
    if not (os.path.exists(lvbert_train_oof_path) and os.path.exists(lvbert_val_path) and os.path.exists(lvbert_test_path)):
        print("\nLVBERT files not found. Running LVBERT first because HSSC needs them...")
        run_lvbert(train_df, val_df, test_df, y_train, y_val, y_test,
                   numeric_cols, categorical_cols, label_encoder, feature_mode)

    lvbert_train_oof = pd.read_csv(lvbert_train_oof_path)
    lvbert_val = pd.read_csv(lvbert_val_path)
    lvbert_test = pd.read_csv(lvbert_test_path)

    def check_alignment(base_df, pred_df, split_name):
        if len(base_df) != len(pred_df):
            raise ValueError(f"{split_name}: row count mismatch")

        for col in ["article_id", "sentence_id"]:
            if col in base_df.columns and col in pred_df.columns:
                same = (base_df[col].astype(str).values == pred_df[col].astype(str).values).all()
                if not same:
                    raise ValueError(f"{split_name}: row alignment mismatch in column {col}")

    check_alignment(train_df, lvbert_train_oof, "train")
    check_alignment(val_df, lvbert_val, "validation")
    check_alignment(test_df, lvbert_test, "test")

    # -----------------------------
    # IMPORTANT FIX:
    # train uses *_cv columns
    # val/test use normal columns
    # -----------------------------
    train_prob_cols = [c for c in lvbert_train_oof.columns if c.startswith("pred_prob_") and c.endswith("_cv")]
    val_prob_cols = [c for c in lvbert_val.columns if c.startswith("pred_prob_") and not c.endswith("_cv")]
    test_prob_cols = [c for c in lvbert_test.columns if c.startswith("pred_prob_") and not c.endswith("_cv")]

    if not train_prob_cols:
        raise ValueError("No pred_prob_*_cv columns found in LVBERT train OOF file.")

    if not val_prob_cols:
        raise ValueError("No pred_prob_* columns found in LVBERT validation file.")

    if not test_prob_cols:
        raise ValueError("No pred_prob_* columns found in LVBERT test file.")

    # Make common semantic column names for HSSC
    semantic_base_names = [c.replace("_cv", "") for c in train_prob_cols]

    def merge_semantic_features(base_df, pred_df, prob_cols, has_cv_confidence=False):
        out = base_df.copy()

        for source_col, target_col in zip(prob_cols, semantic_base_names):
            out[target_col] = pred_df[source_col].values

        if has_cv_confidence and "pred_confidence_cv" in pred_df.columns:
            out["lvbert_pred_confidence"] = pred_df["pred_confidence_cv"].values
        elif (not has_cv_confidence) and "pred_confidence" in pred_df.columns:
            out["lvbert_pred_confidence"] = pred_df["pred_confidence"].values

        return out

    train_feat_df = merge_semantic_features(train_df, lvbert_train_oof, train_prob_cols, has_cv_confidence=True)
    val_feat_df = merge_semantic_features(val_df, lvbert_val, val_prob_cols, has_cv_confidence=False)
    test_feat_df = merge_semantic_features(test_df, lvbert_test, test_prob_cols, has_cv_confidence=False)

    semantic_cols = semantic_base_names + (
        ["lvbert_pred_confidence"] if "lvbert_pred_confidence" in train_feat_df.columns else []
    )

    numeric_cols_local = [c for c in numeric_cols if c in train_feat_df.columns]
    categorical_cols_local = [c for c in categorical_cols if c in train_feat_df.columns]

    feature_cols = semantic_cols + numeric_cols_local + categorical_cols_local

    X_train = train_feat_df[feature_cols].copy()
    X_val = val_feat_df[feature_cols].copy()
    X_test = test_feat_df[feature_cols].copy()

    groups_train = train_feat_df[GROUP_COL].values

    def build_hssc_model():
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", SkPipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]), semantic_cols + numeric_cols_local),
                ("cat", SkPipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]), categorical_cols_local),
            ],
            remainder="drop"
        )

        clf = LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            multi_class="auto",
            solver="lbfgs",
            random_state=SEED
        )

        return SkPipeline([
            ("preprocessor", preprocessor),
            ("clf", clf)
        ])

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    n_classes = len(label_encoder.classes_)
    oof_probs = np.zeros((len(train_feat_df), n_classes), dtype=np.float32)
    oof_preds = np.zeros(len(train_feat_df), dtype=np.int32)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups=groups_train), start=1):
        print_header(f"HSSC FOLD {fold}/5")

        model = build_hssc_model()
        model.fit(X_train.iloc[tr_idx], y_train[tr_idx])

        fold_probs = model.predict_proba(X_train.iloc[va_idx])
        fold_preds = np.argmax(fold_probs, axis=1)

        oof_probs[va_idx] = fold_probs
        oof_preds[va_idx] = fold_preds

        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(y_train[va_idx], fold_preds),
            "macro_f1": f1_score(y_train[va_idx], fold_preds, average="macro"),
            "weighted_f1": f1_score(y_train[va_idx], fold_preds, average="weighted")
        })

        print_eval(y_train[va_idx], fold_preds, label_encoder, f"HSSC | Fold {fold} validation")

    print_eval(y_train, oof_preds, label_encoder, "HSSC | 5-fold CV on TRAIN")

    final_model = build_hssc_model()
    final_model.fit(X_train, y_train)

    val_probs = final_model.predict_proba(X_val)
    val_preds = np.argmax(val_probs, axis=1)
    print_eval(y_val, val_preds, label_encoder, "HSSC | VALIDATION")

    test_probs = final_model.predict_proba(X_test)
    test_preds = np.argmax(test_probs, axis=1)
    print_eval(y_test, test_preds, label_encoder, "HSSC | TEST")

    pd.DataFrame(fold_rows).to_csv(fold_metrics_path, index=False, encoding="utf-8-sig")

    save_multiclass_prediction_file(train_feat_df, oof_probs, oof_preds, train_oof_path, label_encoder, cv_mode=True)
    save_multiclass_prediction_file(val_feat_df, val_probs, val_preds, val_pred_path, label_encoder, cv_mode=False)
    save_multiclass_prediction_file(test_feat_df, test_probs, test_preds, test_pred_path, label_encoder, cv_mode=False)

    save_summary({
        "model": f"HSSC_{feature_mode}",
        "cv_accuracy": accuracy_score(y_train, oof_preds),
        "cv_macro_f1": f1_score(y_train, oof_preds, average="macro"),
        "cv_weighted_f1": f1_score(y_train, oof_preds, average="weighted"),
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_macro_f1": f1_score(y_val, val_preds, average="macro"),
        "val_weighted_f1": f1_score(y_val, val_preds, average="weighted"),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_macro_f1": f1_score(y_test, test_preds, average="macro"),
        "test_weighted_f1": f1_score(y_test, test_preds, average="weighted")
    }, summary_path)

    print("\nSaved files:")
    print(train_oof_path)
    print(val_pred_path)
    print(test_pred_path)
    print(fold_metrics_path)
    print(summary_path)

    return {
        "train_oof_path": train_oof_path,
        "val_pred_path": val_pred_path,
        "test_pred_path": test_pred_path,
        "summary_path": summary_path
    }

# =========================================================
# 9. RUN FUNCTION
# =========================================================
def run_experiment(model_type="svm", feature_mode="legacy"):
    reset_seeds(SEED)

    if model_type not in ["svm", "lstm", "lvbert", "hssc"]:
        raise ValueError("model_type must be one of: svm, lstm, lvbert, hssc")

    if feature_mode not in ["legacy", "safe"]:
        raise ValueError("feature_mode must be one of: legacy, safe")

    print_header(f"RUNNING {model_type.upper()} | feature_mode={feature_mode}")

    train_df, val_df, test_df, numeric_cols, categorical_cols = load_data(feature_mode)
    label_encoder, y_train, y_val, y_test = encode_labels(train_df, val_df, test_df)

    print("Numeric cols:", numeric_cols)
    print("Categorical cols:", categorical_cols)
    print("Role labels:", list(label_encoder.classes_))
    print("Train articles:", train_df[GROUP_COL].nunique(), "sentences:", len(train_df))
    print("Val articles:", val_df[GROUP_COL].nunique(), "sentences:", len(val_df))
    print("Test articles:", test_df[GROUP_COL].nunique(), "sentences:", len(test_df))

    if model_type == "svm":
        return run_svm(
            train_df, val_df, test_df,
            y_train, y_val, y_test,
            numeric_cols, categorical_cols,
            label_encoder, feature_mode
        )

    elif model_type == "lstm":
        return run_lstm(
            train_df, val_df, test_df,
            y_train, y_val, y_test,
            numeric_cols, categorical_cols,
            label_encoder, feature_mode
        )

    elif model_type == "lvbert":
        return run_lvbert(
            train_df, val_df, test_df,
            y_train, y_val, y_test,
            numeric_cols, categorical_cols,
            label_encoder, feature_mode
        )

    elif model_type == "hssc":
        return run_hssc(
            train_df, val_df, test_df,
            y_train, y_val, y_test,
            numeric_cols, categorical_cols,
            label_encoder, feature_mode
        )

# SVM

In [ ]:
run_experiment("svm", "legacy")


RUNNING SVM | feature_mode=legacy
Numeric cols: ['sentence_id', 'sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len', 'claim_group_id', 'linked_claim_sentence_id', 'is_primary_claim']
Categorical cols: ['position_bucket', 'source_part', 'sample_group', 'group']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

SVM FOLD 1/5

===== SVM | Fold 1 validation =====
Accuracy: 0.9096
Macro F1: 0.6595
Weighted F1: 0.9003
                  precision    recall  f1-score   support

      BACKGROUND     1.0000    1.0000    1.0000       237
           CLAIM     0.8824    0.6742    0.7643        89
 EVIDENCE_REFUTE     0.7243    0.9394    0.8179       165
EVIDENCE_SUPPORT     1.0000    1.0000    1.0000       243
  VERDICT_REFUTE     0.8182    0.2432    0.3750        37
 V

{'train_oof_path': '/content/universal_sentence_experiments/svm_legacy/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/svm_legacy/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/svm_legacy/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/svm_legacy/summary.csv'}

# LSTM

In [ ]:
run_experiment("lstm", "legacy")


RUNNING LSTM | feature_mode=legacy
Numeric cols: ['sentence_id', 'sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len', 'claim_group_id', 'linked_claim_sentence_id', 'is_primary_claim']
Categorical cols: ['position_bucket', 'source_part', 'sample_group', 'group']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

LSTM FOLD 1/5

===== LSTM | Fold 1 validation =====
Accuracy: 0.7106
Macro F1: 0.4762
Weighted F1: 0.6909
                  precision    recall  f1-score   support

      BACKGROUND     0.6905    0.8565    0.7646       237
           CLAIM     1.0000    0.5169    0.6815        89
 EVIDENCE_REFUTE     0.5928    0.6970    0.6407       165
EVIDENCE_SUPPORT     0.7750    0.7654    0.7702       243
  VERDICT_REFUTE     0.0000    0.0000    0.0000        37

{'train_oof_path': '/content/universal_sentence_experiments/lstm_legacy/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/lstm_legacy/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/lstm_legacy/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/lstm_legacy/summary.csv'}

# LVBERT

In [ ]:
run_experiment("lvbert", "legacy")


RUNNING LVBERT | feature_mode=legacy
Numeric cols: ['sentence_id', 'sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len', 'claim_group_id', 'linked_claim_sentence_id', 'is_primary_claim']
Categorical cols: ['position_bucket', 'source_part', 'sample_group', 'group']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

LVBERT FOLD 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: AiLab-IMCS-UL/lvbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== LVBERT | Fold 1 validation =====
Accuracy: 0.8708
Macro F1: 0.6597
Weighted F1: 0.8645
                  precision    recall  f1-score   support

      BACKGROUND     0.8934    0.9198    0.9064       237
           CLAIM     0.9365    0.6629    0.7763        89
 EVIDENCE_REFUTE     0.7735    0.8485    0.8092       165
EVIDENCE_SUPPORT     0.9269    0.9918    0.9583       243
  VERDICT_REFUTE     0.6154    0.4324    0.5079        37
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         3

        accuracy                         0.8708       774
       macro avg     0.6910    0.6426    0.6597       774
    weighted avg     0.8666    0.8708    0.8645       774


LVBERT FOLD 2/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


===== LVBERT | Fold 5 validation =====
Accuracy: 0.8223
Macro F1: 0.6117
Weighted F1: 0.8186
                  precision    recall  f1-score   support

      BACKGROUND     0.9562    0.7860    0.8628       472
           CLAIM     0.9265    0.6495    0.7636        97
 EVIDENCE_REFUTE     0.7155    0.9291    0.8084       268
EVIDENCE_SUPPORT     0.7735    0.9373    0.8475       255
  VERDICT_REFUTE     0.5417    0.3023    0.3881        43
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         2

        accuracy                         0.8223      1137
       macro avg     0.6522    0.6007    0.6117      1137
    weighted avg     0.8386    0.8223    0.8186      1137


===== LVBERT | 5-fold CV on TRAIN =====
Accuracy: 0.8398
Macro F1: 0.6205
Weighted F1: 0.8362
                  precision    recall  f1-score   support

      BACKGROUND     0.9115    0.9100    0.9107      1923
           CLAIM     0.6695    0.7682    0.7155       509
 EVIDENCE_REFUTE     0.7864    0.8414    0.8130      

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: AiLab-IMCS-UL/lvbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== LVBERT | VALIDATION =====
Accuracy: 0.8896
Macro F1: 0.6296
Weighted F1: 0.8793
                  precision    recall  f1-score   support

      BACKGROUND     0.9624    0.9597    0.9610       347
           CLAIM     0.7857    0.6947    0.7374        95
 EVIDENCE_REFUTE     0.8317    0.9245    0.8756       278
EVIDENCE_SUPPORT     0.9072    0.9513    0.9287       226
  VERDICT_REFUTE     0.6364    0.1750    0.2745        40
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         1

        accuracy                         0.8896       987
       macro avg     0.6872    0.6175    0.6296       987
    weighted avg     0.8818    0.8896    0.8793       987


===== LVBERT | TEST =====
Accuracy: 0.8921
Macro F1: 0.6672
Weighted F1: 0.8898
                  precision    recall  f1-score   support

      BACKGROUND     0.9725    0.9043    0.9372       470
           CLAIM     0.8000    0.8276    0.8136        87
 EVIDENCE_REFUTE     0.7839    0.9017    0.8387       173
EVIDENCE_SUPPORT

{'train_oof_path': '/content/universal_sentence_experiments/lvbert_legacy/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/lvbert_legacy/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/lvbert_legacy/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/lvbert_legacy/summary.csv'}

# LVBERT

In [ ]:
run_experiment("hssc", "legacy")


RUNNING HSSC | feature_mode=legacy
Numeric cols: ['sentence_id', 'sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len', 'claim_group_id', 'linked_claim_sentence_id', 'is_primary_claim']
Categorical cols: ['position_bucket', 'source_part', 'sample_group', 'group']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

HSSC FOLD 1/5

===== HSSC | Fold 1 validation =====
Accuracy: 0.8553
Macro F1: 0.6689
Weighted F1: 0.8764
                  precision    recall  f1-score   support

      BACKGROUND     0.9916    0.9958    0.9937       237
           CLAIM     0.7805    0.7191    0.7485        89
 EVIDENCE_REFUTE     0.9029    0.5636    0.6940       165
EVIDENCE_SUPPORT     1.0000    1.0000    1.0000       243
  VERDICT_REFUTE     0.4068    0.6486    0.5000        37

{'train_oof_path': '/content/universal_sentence_experiments/hssc_legacy/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/hssc_legacy/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/hssc_legacy/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/hssc_legacy/summary.csv'}

# Experiment without structural features

In [ ]:
run_experiment("svm", "safe")


RUNNING SVM | feature_mode=safe
Numeric cols: ['sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len']
Categorical cols: ['position_bucket', 'source_part']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

SVM FOLD 1/5

===== SVM | Fold 1 validation =====
Accuracy: 0.4858
Macro F1: 0.3428
Weighted F1: 0.467
                  precision    recall  f1-score   support

      BACKGROUND     0.4090    0.6160    0.4916       237
           CLAIM     0.7778    0.1573    0.2617        89
 EVIDENCE_REFUTE     0.4958    0.3576    0.4155       165
EVIDENCE_SUPPORT     0.5535    0.6173    0.5837       243
  VERDICT_REFUTE     0.7778    0.1892    0.3043        37
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         3

        accuracy                         0.4858     

{'train_oof_path': '/content/universal_sentence_experiments/svm_safe/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/svm_safe/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/svm_safe/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/svm_safe/summary.csv'}

In [ ]:
run_experiment("hssc", "safe")


RUNNING HSSC | feature_mode=safe
Numeric cols: ['sentence_index_in_article', 'num_sentences_in_article', 'relative_position', 'article_word_count', 'sentence_char_len']
Categorical cols: ['position_bucket', 'source_part']
Role labels: ['BACKGROUND', 'CLAIM', 'EVIDENCE_REFUTE', 'EVIDENCE_SUPPORT', 'VERDICT_REFUTE', 'VERDICT_SUPPORT']
Train articles: 207 sentences: 5494
Val articles: 45 sentences: 987
Test articles: 45 sentences: 982

LVBERT files not found. Running LVBERT first because HSSC needs them...

LVBERT FOLD 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: AiLab-IMCS-UL/lvbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== LVBERT | Fold 1 validation =====
Accuracy: 0.5956
Macro F1: 0.4176
Weighted F1: 0.582
                  precision    recall  f1-score   support

      BACKGROUND     0.5018    0.5907    0.5426       237
           CLAIM     0.6087    0.3146    0.4148        89
 EVIDENCE_REFUTE     0.5475    0.5939    0.5698       165
EVIDENCE_SUPPORT     0.7224    0.7819    0.7510       243
  VERDICT_REFUTE     0.7143    0.1351    0.2273        37
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         3

        accuracy                         0.5956       774
       macro avg     0.5158    0.4027    0.4176       774
    weighted avg     0.6013    0.5956    0.5820       774


LVBERT FOLD 2/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


===== LVBERT | Fold 5 validation =====
Accuracy: 0.5866
Macro F1: 0.4123
Weighted F1: 0.5722
                  precision    recall  f1-score   support

      BACKGROUND     0.6327    0.5000    0.5586       472
           CLAIM     0.5652    0.4021    0.4699        97
 EVIDENCE_REFUTE     0.5589    0.6903    0.6177       268
EVIDENCE_SUPPORT     0.5655    0.7961    0.6612       255
  VERDICT_REFUTE     0.8000    0.0930    0.1667        43
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         2

        accuracy                         0.5866      1137
       macro avg     0.5204    0.4136    0.4123      1137
    weighted avg     0.5997    0.5866    0.5722      1137


===== LVBERT | 5-fold CV on TRAIN =====
Accuracy: 0.5759
Macro F1: 0.4341
Weighted F1: 0.5685
                  precision    recall  f1-score   support

      BACKGROUND     0.5415    0.5226    0.5319      1923
           CLAIM     0.5827    0.4361    0.4989       509
 EVIDENCE_REFUTE     0.5647    0.6156    0.5890      

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: AiLab-IMCS-UL/lvbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== LVBERT | VALIDATION =====
Accuracy: 0.5755
Macro F1: 0.384
Weighted F1: 0.5569
                  precision    recall  f1-score   support

      BACKGROUND     0.5600    0.6859    0.6166       347
           CLAIM     0.8000    0.2105    0.3333        95
 EVIDENCE_REFUTE     0.5719    0.6583    0.6120       278
EVIDENCE_SUPPORT     0.5775    0.5442    0.5604       226
  VERDICT_REFUTE     1.0000    0.1000    0.1818        40
 VERDICT_SUPPORT     0.0000    0.0000    0.0000         1

        accuracy                         0.5755       987
       macro avg     0.5849    0.3665    0.3840       987
    weighted avg     0.6077    0.5755    0.5569       987


===== LVBERT | TEST =====
Accuracy: 0.5479
Macro F1: 0.4009
Weighted F1: 0.5425
                  precision    recall  f1-score   support

      BACKGROUND     0.6082    0.5681    0.5875       470
           CLAIM     0.7429    0.2989    0.4262        87
 EVIDENCE_REFUTE     0.4563    0.5434    0.4960       173
EVIDENCE_SUPPORT 

{'train_oof_path': '/content/universal_sentence_experiments/hssc_safe/train_oof_predictions.csv',
 'val_pred_path': '/content/universal_sentence_experiments/hssc_safe/validation_predictions.csv',
 'test_pred_path': '/content/universal_sentence_experiments/hssc_safe/test_predictions.csv',
 'summary_path': '/content/universal_sentence_experiments/hssc_safe/summary.csv'}

# Claim classification

In [ ]:
# =========================================================
# CLAIM-GROUP EVALUATION (True vs false claim)
# Uses saved sentence-level predictions
# Works for: svm, lstm, lvbert, hssc
# Modes: legacy / safe
# =========================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# =========================================================
# 1. CONFIG
# =========================================================
TEST_GOLD_PATH = "/content/test_sen.csv"

def get_experiment_dir(model_type, feature_mode):
    return os.path.join("/content/universal_sentence_experiments", f"{model_type}_{feature_mode}")

# =========================================================
# 2. HELPERS
# =========================================================
def print_header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

def derive_gold_decision(group_df):
    roles = set(group_df["role_label"].astype(str).tolist())
    classes = set(pd.to_numeric(group_df["class_label"], errors="coerce").dropna().astype(int).tolist())

    if "CLAIM" not in roles:
        return "no_claim"

    if 1 in classes:
        return "misleading_false_claim"
    elif 0 in classes:
        return "supported_true_claim"
    else:
        return "unresolved_insufficient_evidence"

def decide_group_hard(group_df):
    labels = set(group_df["pred_role_label"].astype(str).tolist())

    if "CLAIM" not in labels:
        return "no_claim"

    has_refute = ("VERDICT_REFUTE" in labels) or ("EVIDENCE_REFUTE" in labels)
    has_support = ("VERDICT_SUPPORT" in labels) or ("EVIDENCE_SUPPORT" in labels)

    if has_refute:
        return "misleading_false_claim"
    elif has_support:
        return "supported_true_claim"
    else:
        return "unresolved_insufficient_evidence"

# =========================================================
# 3. MAIN FUNCTION
# =========================================================
def run_claim_group_experiment(model_type="svm", feature_mode="legacy"):
    if model_type not in ["svm", "lstm", "lvbert", "hssc"]:
        raise ValueError("model_type must be one of: svm, lstm, lvbert, hssc")

    if feature_mode not in ["legacy", "safe"]:
        raise ValueError("feature_mode must be one of: legacy, safe")

    print_header(f"RUNNING CLAIM-GROUP EVALUATION | {model_type.upper()} | {feature_mode}")

    exp_dir = get_experiment_dir(model_type, feature_mode)

    pred_path = os.path.join(exp_dir, "test_predictions.csv")
    hard_decisions_out = os.path.join(exp_dir, "claim_group_final_decisions_hard.csv")
    hard_eval_out = os.path.join(exp_dir, "claim_group_eval_hard.csv")
    hard_summary_out = os.path.join(exp_dir, "claim_group_summary_hard.csv")

    if not os.path.exists(pred_path):
        raise ValueError(
            f"Prediction file not found:\n{pred_path}\n\n"
            f"Vispirms palaid Kods 1:\nrun_experiment('{model_type}', '{feature_mode}')"
        )

    gold_df = pd.read_csv(TEST_GOLD_PATH)
    pred_df = pd.read_csv(pred_path)

    for df in [gold_df, pred_df]:
        for col in ["article_id", "claim_group_id", "sentence_id"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    gold_df = gold_df.dropna(subset=["article_id", "claim_group_id"]).copy()
    pred_df = pred_df.dropna(subset=["article_id", "claim_group_id"]).copy()

    gold_df["article_id"] = gold_df["article_id"].astype(int)
    gold_df["claim_group_id"] = gold_df["claim_group_id"].astype(int)
    pred_df["article_id"] = pred_df["article_id"].astype(int)
    pred_df["claim_group_id"] = pred_df["claim_group_id"].astype(int)

    # -----------------------------------------------------
    # GOLD GROUP TABLE
    # -----------------------------------------------------
    gold_rows = []
    for (article_id, claim_group_id), g in gold_df.groupby(["article_id", "claim_group_id"]):
        gold_rows.append({
            "article_id": article_id,
            "claim_group_id": claim_group_id,
            "gold_decision": derive_gold_decision(g),
            "gold_num_sentences": len(g),
            "gold_roles_in_group": ", ".join(sorted(set(g["role_label"].astype(str))))
        })

    gold_group_df = pd.DataFrame(gold_rows)

    print("Gold claim groups:", len(gold_group_df))
    print(gold_group_df["gold_decision"].value_counts(dropna=False).to_dict())

    # -----------------------------------------------------
    # PREDICTED GROUP TABLE
    # -----------------------------------------------------
    pred_rows = []
    for (article_id, claim_group_id), g in pred_df.groupby(["article_id", "claim_group_id"]):
        pred_rows.append({
            "article_id": article_id,
            "claim_group_id": claim_group_id,
            "final_decision": decide_group_hard(g),
            "num_sentences": len(g),
            "predicted_labels_in_group": ", ".join(sorted(set(g["pred_role_label"].astype(str))))
        })

    hard_df = pd.DataFrame(pred_rows)
    hard_df.to_csv(hard_decisions_out, index=False, encoding="utf-8-sig")

    # -----------------------------------------------------
    # MERGE + EVALUATE
    # -----------------------------------------------------
    merged = gold_group_df.merge(
        hard_df,
        on=["article_id", "claim_group_id"],
        how="left"
    )

    merged["final_decision"] = merged["final_decision"].fillna("no_claim")

    eval_df = merged[
        merged["gold_decision"].isin(["supported_true_claim", "misleading_false_claim"])
    ].copy()

    gold_map = {
        "supported_true_claim": 0,
        "misleading_false_claim": 1
    }

    pred_map = {
        "supported_true_claim": 0,
        "misleading_false_claim": 1
    }

    eval_df["y_true"] = eval_df["gold_decision"].map(gold_map)
    eval_df["y_pred_raw"] = eval_df["final_decision"].map(pred_map)

    # unresolved / no_claim are counted as wrong
    eval_df["y_pred"] = eval_df["y_pred_raw"]
    mask_missing = eval_df["y_pred"].isna()
    eval_df.loc[mask_missing, "y_pred"] = 1 - eval_df.loc[mask_missing, "y_true"]
    eval_df["y_pred"] = eval_df["y_pred"].astype(int)

    acc = accuracy_score(eval_df["y_true"], eval_df["y_pred"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        eval_df["y_true"],
        eval_df["y_pred"],
        average="binary",
        zero_division=0
    )
    cm = confusion_matrix(eval_df["y_true"], eval_df["y_pred"], labels=[0, 1])

    metrics_df = pd.DataFrame({
        "metric": ["accuracy", "precision", "recall", "f1_score"],
        "value": [acc, prec, rec, f1]
    })

    print(f"\n===== {model_type.upper()} END-TO-END (HARD) | {feature_mode} =====")
    print(metrics_df.to_string(index=False))

    print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nClassification report:")
    print(classification_report(
        eval_df["y_true"],
        eval_df["y_pred"],
        target_names=["supported_true_claim", "misleading_false_claim"],
        digits=4,
        zero_division=0
    ))

    print("\nGold label distribution:")
    print(eval_df["gold_decision"].value_counts().to_dict())

    print("\nPredicted decision distribution:")
    print(eval_df["final_decision"].value_counts().to_dict())

    eval_df.to_csv(hard_eval_out, index=False, encoding="utf-8-sig")

    summary_df = pd.DataFrame([{
        "model": f"{model_type}_end_to_end_hard_{feature_mode}",
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }])
    summary_df.to_csv(hard_summary_out, index=False, encoding="utf-8-sig")

    print("\nSaved files:")
    print(hard_decisions_out)
    print(hard_eval_out)
    print(hard_summary_out)

    return {
        "decisions_path": hard_decisions_out,
        "eval_path": hard_eval_out,
        "summary_path": hard_summary_out
    }

SVM

In [ ]:
run_claim_group_experiment("svm", "legacy")


RUNNING CLAIM-GROUP EVALUATION | SVM | legacy
Gold claim groups: 97
{'no_claim': 41, 'misleading_false_claim': 34, 'supported_true_claim': 22}

===== SVM END-TO-END (HARD) | legacy =====
   metric    value
 accuracy 0.928571
precision 0.968750
   recall 0.911765
 f1_score 0.939394

Confusion matrix [[TN, FP], [FN, TP]]:
[[21  1]
 [ 3 31]]

Classification report:
                        precision    recall  f1-score   support

  supported_true_claim     0.8750    0.9545    0.9130        22
misleading_false_claim     0.9688    0.9118    0.9394        34

              accuracy                         0.9286        56
             macro avg     0.9219    0.9332    0.9262        56
          weighted avg     0.9319    0.9286    0.9290        56


Gold label distribution:
{'misleading_false_claim': 34, 'supported_true_claim': 22}

Predicted decision distribution:
{'misleading_false_claim': 31, 'supported_true_claim': 21, 'unresolved_insufficient_evidence': 2, 'no_claim': 2}

Saved files:
/

{'decisions_path': '/content/universal_sentence_experiments/svm_legacy/claim_group_final_decisions_hard.csv',
 'eval_path': '/content/universal_sentence_experiments/svm_legacy/claim_group_eval_hard.csv',
 'summary_path': '/content/universal_sentence_experiments/svm_legacy/claim_group_summary_hard.csv'}

LSTM

In [ ]:
run_claim_group_experiment("lstm", "legacy")


RUNNING CLAIM-GROUP EVALUATION | LSTM | legacy
Gold claim groups: 97
{'no_claim': 41, 'misleading_false_claim': 34, 'supported_true_claim': 22}

===== LSTM END-TO-END (HARD) | legacy =====
   metric    value
 accuracy 0.589286
precision 0.617021
   recall 0.852941
 f1_score 0.716049

Confusion matrix [[TN, FP], [FN, TP]]:
[[ 4 18]
 [ 5 29]]

Classification report:
                        precision    recall  f1-score   support

  supported_true_claim     0.4444    0.1818    0.2581        22
misleading_false_claim     0.6170    0.8529    0.7160        34

              accuracy                         0.5893        56
             macro avg     0.5307    0.5174    0.4871        56
          weighted avg     0.5492    0.5893    0.5361        56


Gold label distribution:
{'misleading_false_claim': 34, 'supported_true_claim': 22}

Predicted decision distribution:
{'misleading_false_claim': 46, 'supported_true_claim': 4, 'no_claim': 4, 'unresolved_insufficient_evidence': 2}

Saved files:


{'decisions_path': '/content/universal_sentence_experiments/lstm_legacy/claim_group_final_decisions_hard.csv',
 'eval_path': '/content/universal_sentence_experiments/lstm_legacy/claim_group_eval_hard.csv',
 'summary_path': '/content/universal_sentence_experiments/lstm_legacy/claim_group_summary_hard.csv'}

LVBERT

In [ ]:
run_claim_group_experiment("lvbert", "legacy")


RUNNING CLAIM-GROUP EVALUATION | LVBERT | legacy
Gold claim groups: 97
{'no_claim': 41, 'misleading_false_claim': 34, 'supported_true_claim': 22}

===== LVBERT END-TO-END (HARD) | legacy =====
   metric    value
 accuracy 0.964286
precision 0.970588
   recall 0.970588
 f1_score 0.970588

Confusion matrix [[TN, FP], [FN, TP]]:
[[21  1]
 [ 1 33]]

Classification report:
                        precision    recall  f1-score   support

  supported_true_claim     0.9545    0.9545    0.9545        22
misleading_false_claim     0.9706    0.9706    0.9706        34

              accuracy                         0.9643        56
             macro avg     0.9626    0.9626    0.9626        56
          weighted avg     0.9643    0.9643    0.9643        56


Gold label distribution:
{'misleading_false_claim': 34, 'supported_true_claim': 22}

Predicted decision distribution:
{'misleading_false_claim': 33, 'supported_true_claim': 21, 'unresolved_insufficient_evidence': 2}

Saved files:
/content/u

{'decisions_path': '/content/universal_sentence_experiments/lvbert_legacy/claim_group_final_decisions_hard.csv',
 'eval_path': '/content/universal_sentence_experiments/lvbert_legacy/claim_group_eval_hard.csv',
 'summary_path': '/content/universal_sentence_experiments/lvbert_legacy/claim_group_summary_hard.csv'}

HSSC

In [ ]:
run_claim_group_experiment("hssc", "legacy")


RUNNING CLAIM-GROUP EVALUATION | HSSC | legacy
Gold claim groups: 97
{'no_claim': 41, 'misleading_false_claim': 34, 'supported_true_claim': 22}

===== HSSC END-TO-END (HARD) | legacy =====
   metric    value
 accuracy 0.964286
precision 0.970588
   recall 0.970588
 f1_score 0.970588

Confusion matrix [[TN, FP], [FN, TP]]:
[[21  1]
 [ 1 33]]

Classification report:
                        precision    recall  f1-score   support

  supported_true_claim     0.9545    0.9545    0.9545        22
misleading_false_claim     0.9706    0.9706    0.9706        34

              accuracy                         0.9643        56
             macro avg     0.9626    0.9626    0.9626        56
          weighted avg     0.9643    0.9643    0.9643        56


Gold label distribution:
{'misleading_false_claim': 34, 'supported_true_claim': 22}

Predicted decision distribution:
{'misleading_false_claim': 33, 'supported_true_claim': 21, 'unresolved_insufficient_evidence': 2}

Saved files:
/content/unive

{'decisions_path': '/content/universal_sentence_experiments/hssc_legacy/claim_group_final_decisions_hard.csv',
 'eval_path': '/content/universal_sentence_experiments/hssc_legacy/claim_group_eval_hard.csv',
 'summary_path': '/content/universal_sentence_experiments/hssc_legacy/claim_group_summary_hard.csv'}

In [ ]:
import os
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

BASE_DIR = "/content/universal_sentence_experiments"
FEATURE_MODE = "legacy"
MODELS = ["svm", "lstm", "lvbert", "hssc"]

TEST_GOLD_PATH = "/content/test_sen.csv"
CLAIM_LABEL = "CLAIM"

gold_df = pd.read_csv(TEST_GOLD_PATH)

rows = []

for model in MODELS:
    summary_path = os.path.join(BASE_DIR, f"{model}_{FEATURE_MODE}", "summary.csv")
    pred_path = os.path.join(BASE_DIR, f"{model}_{FEATURE_MODE}", "test_predictions.csv")

    if not os.path.exists(summary_path):
        print(f"Missing summary: {summary_path}")
        continue

    df = pd.read_csv(summary_path)
    row = df.iloc[0].to_dict()

    claim_precision = row.get("claim_precision")
    claim_recall = row.get("claim_recall")
    claim_f1 = row.get("claim_f1")

    # If not already present in summary.csv, compute from prediction file
    if (claim_precision is None or pd.isna(claim_precision)) and os.path.exists(pred_path):
        pred_df = pd.read_csv(pred_path)

        if len(pred_df) != len(gold_df):
            print(f"Length mismatch for {model}: gold={len(gold_df)}, pred={len(pred_df)}")
            claim_precision, claim_recall, claim_f1 = None, None, None
        else:
            y_true = (gold_df["role_label"].astype(str) == CLAIM_LABEL).astype(int)
            y_pred = (pred_df["pred_role_label"].astype(str) == CLAIM_LABEL).astype(int)

            p, r, f, _ = precision_recall_fscore_support(
                y_true,
                y_pred,
                average="binary",
                zero_division=0
            )

            claim_precision = p
            claim_recall = r
            claim_f1 = f

    rows.append({
        "Model": model.upper(),
        "Accuracy": row.get("test_accuracy"),
        "Macro F1": row.get("test_macro_f1"),
        "Weighted F1": row.get("test_weighted_f1"),
        "CLAIM Precision": claim_precision,
        "CLAIM Recall": claim_recall,
        "CLAIM F1": claim_f1
    })

sentence_level_comparison = pd.DataFrame(rows)
sentence_level_comparison = sentence_level_comparison.sort_values("Model").reset_index(drop=True)

print("\n===== KODS 1 COMPARISON TABLE =====")
print(sentence_level_comparison.round(4))

out_path = f"/content/kods1_sentence_level_comparison_{FEATURE_MODE}.csv"
sentence_level_comparison.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(out_path)


===== KODS 1 COMPARISON TABLE =====
    Model  Accuracy  Macro F1  Weighted F1  CLAIM Precision  CLAIM Recall  \
0    HSSC    0.9175    0.6867       0.9226           0.7717        0.8161   
1    LSTM    0.7373    0.5777       0.7272           0.9630        0.5977   
2  LVBERT    0.8921    0.8006       0.8898           0.8000        0.8276   
3     SVM    0.9196    0.7806       0.9148           0.8923        0.6667   

   CLAIM F1  
0    0.7933  
1    0.7376  
2    0.8136  
3    0.7632  

Saved:
/content/kods1_sentence_level_comparison_legacy.csv


In [ ]:
import os
import pandas as pd

BASE_DIR = "/content/universal_sentence_experiments"
FEATURE_MODE = "legacy"
MODELS = ["svm", "lstm", "lvbert", "hssc"]

rows = []

for model in MODELS:
    summary_path = os.path.join(
        BASE_DIR,
        f"{model}_{FEATURE_MODE}",
        "claim_group_summary_hard.csv"
    )

    if not os.path.exists(summary_path):
        print(f"Missing: {summary_path}")
        continue

    df = pd.read_csv(summary_path)
    row = df.iloc[0].to_dict()

    rows.append({
        "Model": model.upper(),
        "Accuracy": row.get("accuracy"),
        "Precision": row.get("precision"),
        "Recall": row.get("recall"),
        "F1": row.get("f1")
    })

end_to_end_comparison = pd.DataFrame(rows)
end_to_end_comparison = end_to_end_comparison.sort_values("Model").reset_index(drop=True)

print("\n===== KODS 3 COMPARISON TABLE =====")
print(end_to_end_comparison.round(4))

end_to_end_comparison.to_csv(
    f"/content/kods3_end_to_end_comparison_{FEATURE_MODE}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(f"/content/kods3_end_to_end_comparison_{FEATURE_MODE}.csv")


===== KODS 3 COMPARISON TABLE =====
    Model  Accuracy  Precision  Recall      F1
0    HSSC    0.9643     0.9706  0.9706  0.9706
1    LSTM    0.5893     0.6170  0.8529  0.7160
2  LVBERT    0.9643     0.9706  0.9706  0.9706
3     SVM    0.9286     0.9688  0.9118  0.9394

Saved:
/content/kods3_end_to_end_comparison_legacy.csv
